# Assignment 1.2: RAG-Based Intelligent Document Assistant (LangChain + LlamaIndex)

Ask questions over your own documents (PDFs, text files, web pages) and get **grounded answers with sources**.

| Stage | Library | What it does |
|---|---|---|
| Ingest | LlamaIndex | `SimpleDirectoryReader` for PDF/TXT, web scraper for URLs |
| Index | LlamaIndex | `SentenceSplitter` chunks -> local embeddings -> `VectorStoreIndex` (saved to disk) |
| Retrieve | LlamaIndex | semantic retriever with top-k and **metadata filters** (search one or many documents) |
| Generate | LangChain | prompt templates + LCEL chains + LLM (Groq / Gemini / OpenAI / Anthropic) |
| Parse | LangChain | `PydanticOutputParser` -> typed answer with cited passages |
| Extras | both | multi-document Q&A, follow-up questions, map-reduce summaries |

```
documents -> chunks -> embeddings -> VectorStoreIndex -> retriever (top-k, filters)
                                                              |
question -> (condense with history) -> retrieved passages -> LangChain prompt -> LLM -> parser -> answer + sources
```
Runs on Jupyter, Google Colab and VS Code.

## 0. Install packages
Run once. If Colab shows a dependency *warning* about `google-colab`, ignore it.

In [ ]:
%pip install -q llama-index-core llama-index-readers-file llama-index-embeddings-huggingface sentence-transformers langchain langchain-core langchain-groq langchain-google-genai langchain-openai langchain-anthropic pypdf beautifulsoup4 requests python-dotenv pandas

## 0.1 Choose the LLM and load the API key
Default is **Groq** (`openai/gpt-oss-20b`). Add `GROQ_API_KEY` in Colab Secrets (key icon, Notebook access ON), or in a `.env` file for Jupyter / VS Code. Change `LLM_PROVIDER` to `gemini`, `openai` or `anthropic` to switch.

In [2]:
import os

LLM_PROVIDER = "groq"                 # groq | gemini | openai | anthropic
LLM_MODEL = "openai/gpt-oss-20b"      # leave "" for the provider default

os.environ["LLM_PROVIDER"] = LLM_PROVIDER
if LLM_MODEL:
    os.environ["LLM_MODEL"] = LLM_MODEL

KEY_NAME = {"groq": "GROQ_API_KEY", "gemini": "GOOGLE_API_KEY",
            "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}[LLM_PROVIDER]

if not os.getenv(KEY_NAME):
    try:                                          # Colab Secrets
        from google.colab import userdata
        os.environ[KEY_NAME] = userdata.get(KEY_NAME)
    except Exception:
        try:                                      # .env file
            from dotenv import load_dotenv
            load_dotenv()
        except Exception:
            pass

print("Provider:", LLM_PROVIDER, "| key loaded:", bool(os.getenv(KEY_NAME)))
if not os.getenv(KEY_NAME):
    print("Key missing: add", KEY_NAME, "to Colab Secrets / .env, or run: os.environ['" + KEY_NAME + "'] = 'your_key'")

Provider: groq | key loaded: True


## 1. Imports, LLM factory (LangChain) and embeddings (LlamaIndex)
- **LlamaIndex**: document loading, chunking, vector index, retriever
- **LangChain**: LLM integration, prompt templates, chains (LCEL), output parsing

In [ ]:
import os
import re
import time
from typing import List, Optional

import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableLambda

from llama_index.core import (Document, VectorStoreIndex, SimpleDirectoryReader, Settings,
                              StorageContext, load_index_from_storage)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.llms import MockLLM
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator

EMBED_MODEL = os.getenv("EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")


def get_llm(temperature: float = 0.1):
    """LangChain chat model chosen by env vars LLM_PROVIDER / LLM_MODEL."""
    provider = os.getenv("LLM_PROVIDER", "groq").lower()
    model = os.getenv("LLM_MODEL", "")
    if provider == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model=model or "openai/gpt-oss-20b", temperature=temperature)   # GROQ_API_KEY
    if provider == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=model or "gemini-3.8-flash", temperature=temperature)  # GOOGLE_API_KEY
    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model or "gpt-4o-mini", temperature=temperature)        # OPENAI_API_KEY
    if provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=model or "claude-haiku-4-5-20251001", temperature=temperature)  # ANTHROPIC_API_KEY
    raise ValueError(f"Unknown LLM_PROVIDER: {provider}")


_EMBED = None


def get_embed_model():
    """Free local embedding model (no API key). Loaded once."""
    global _EMBED
    if _EMBED is None:
        from llama_index.embeddings.huggingface import HuggingFaceEmbedding
        _EMBED = HuggingFaceEmbedding(model_name=EMBED_MODEL)
    return _EMBED


def setup_llamaindex(chunk_size: int = 512, chunk_overlap: int = 64):
    Settings.embed_model = get_embed_model()
    Settings.llm = MockLLM()          # LlamaIndex does retrieval only; LangChain generates the answers
    Settings.node_parser = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)


def run_with_retry(fn, tries: int = 4, wait: int = 20):
    """Retry temporary API errors (503 / 429 rate limits)."""
    for i in range(1, tries + 1):
        try:
            return fn()
        except Exception as e:
            msg = str(e)
            temp = any(x in msg for x in ("503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED", "rate_limit", "overloaded"))
            if temp and i < tries:
                print(f"Temporary API error (attempt {i}/{tries}). Retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

## 2. Ingestion: PDFs / text files / web pages -> LlamaIndex `Document`s

In [ ]:
def _read_pdf(path: str) -> List[Document]:
    """One Document per page. Layout mode keeps each table row on one line ('label | v1 | v2')."""
    from pypdf import PdfReader
    name, docs = os.path.basename(path), []
    for i, page in enumerate(PdfReader(path).pages, 1):
        try:
            text = page.extract_text(extraction_mode="layout")
        except TypeError:                       # very old pypdf
            text = page.extract_text()
        lines = [re.sub(r"\s{2,}", " | ", ln.strip()) for ln in (text or "").splitlines() if ln.strip()]
        if lines:
            docs.append(Document(text="\n".join(lines),
                                 metadata={"file_name": name, "page_label": str(i), "source_type": "pdf"}))
    return docs


def load_files(paths) -> List[Document]:
    """PDFs (page-wise, table-friendly) plus TXT / MD / DOCX via LlamaIndex SimpleDirectoryReader."""
    docs, others = [], []
    for p in paths:
        p = str(p)
        if p.lower().endswith(".pdf"):
            docs.extend(_read_pdf(p))
        else:
            others.append(p)
    if others:
        docs.extend(SimpleDirectoryReader(input_files=others).load_data())
    return docs


def load_web(url: str) -> List[Document]:
    r = requests.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "noscript", "aside"]):
        tag.decompose()
    text = re.sub(r"\n\s*\n+", "\n\n", soup.get_text("\n")).strip()
    title = (soup.title.string.strip() if soup.title and soup.title.string else url)[:80]
    return [Document(text=text, metadata={"file_name": title, "source_type": "web", "url": url})]


SAMPLE_DOCS = {
    "rolling_forecast_guide.txt": """Acme Retail Rolling Forecast Guide.
Acme maintains a 12-month rolling forecast. After every month-end close the forecast is extended by one month, so the horizon always stays at twelve months. The forecast is refreshed on the fifth working day of each month.
The forecast is driver based: revenue equals orders multiplied by average order value (AOV). Sales volume assumptions come from the Category team, marketing spend from the Marketing team, and fulfilment cost per order from the Operations team.
Accuracy targets: total company revenue within 3% of actuals and category level revenue within 5% of actuals.
Version control: every refresh is saved with a version tag such as v-202610. If a refresh moves forecast net revenue by more than 2%, the Finance Director must approve the change before it is published.""",
    "variance_analysis_policy.txt": """Acme Retail Variance Analysis Policy.
Variance is calculated as actual minus budget. A variance is favourable when it increases profit and unfavourable when it reduces profit.
Investigation threshold: any line item that deviates by more than 5% from budget and by more than INR 10 lakh must be investigated.
Variances are split into volume, price and mix effects so that the root cause is clear.
Every investigated variance needs written commentary stating the cause, the financial impact and the action owner. Commentary is due within three working days of the month-end close.
If the same line item shows an unfavourable variance for three months in a row, a re-forecast of that line is triggered.""",
    "ecommerce_kpi_handbook.txt": """Acme Retail E-commerce KPI Handbook.
Average order value (AOV) equals net revenue divided by the number of orders.
Conversion rate equals orders divided by sessions. The internal benchmark is 2.5%.
Customer acquisition cost (CAC) equals marketing spend divided by new customers. The target is below INR 450.
Gross margin equals net revenue minus cost of goods sold, divided by net revenue. Contribution margin also deducts fulfilment, payment gateway and marketing costs; the target is 8%.
The KPI review meeting is held every Monday. The Power BI dashboards are refreshed daily at 6 AM.""",
}


def write_sample_docs(folder: str = "sample_docs") -> List[str]:
    os.makedirs(folder, exist_ok=True)
    paths = []
    for name, text in SAMPLE_DOCS.items():
        p = os.path.join(folder, name)
        with open(p, "w", encoding="utf-8") as f:
            f.write(text)
        paths.append(p)
    return paths

## 3. LangChain prompts, chains and output parsing
- `ChatPromptTemplate` for question condensing, answering, map and reduce summaries
- `PydanticOutputParser` turns the LLM text into a typed `DocAnswer`; if parsing fails we fall back to plain text

In [ ]:
class DocAnswer(BaseModel):
    answer: str = Field(description="Answer using only the context passages. Mention passage numbers like [1].")
    cited_sources: List[int] = Field(description="Numbers of the context passages actually used, e.g. [1, 3]")
    found_in_documents: bool = Field(description="False if the context does not contain the answer")


answer_parser = PydanticOutputParser(pydantic_object=DocAnswer)

QA_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a careful document assistant. Answer ONLY from the numbered context passages. "
     "If the answer is not in the context, say you could not find it in the documents and set found_in_documents to false. "
     "Never use outside knowledge. Text from tables appears as rows like 'label | value | value': match each label only with the values on its own row, and check the column order. "
     "Be concise and mention passage numbers like [1].\n\n{format_instructions}"),
    ("human", "Conversation so far:\n{history}\n\nContext passages:\n{context}\n\nQuestion: {question}"),
]).partial(format_instructions=answer_parser.get_format_instructions())

CONDENSE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the follow-up question as a standalone question using the conversation. "
               "Return only the rewritten question."),
    ("human", "Conversation:\n{history}\n\nFollow-up question: {question}"),
])

MAP_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Summarise the passage in 2-3 sentences. Keep facts and numbers."),
    ("human", "{text}"),
])

REDUCE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Combine the partial summaries of one document into a clear summary: "
               "one overview sentence followed by 4-6 bullet points with the key facts and numbers."),
    ("human", "Document: {name}\n\nPartial summaries:\n{parts}"),
])


def _safe_parse(text: str) -> DocAnswer:
    try:
        return answer_parser.parse(text)
    except Exception:
        return DocAnswer(answer=text.strip(), cited_sources=[], found_in_documents=True)


def build_chains():
    llm = get_llm()
    return {
        "qa": QA_PROMPT | llm | StrOutputParser() | RunnableLambda(_safe_parse),
        "condense": CONDENSE_PROMPT | llm | StrOutputParser(),
        "map": MAP_PROMPT | llm | StrOutputParser(),
        "reduce": REDUCE_PROMPT | llm | StrOutputParser(),
    }

## 4. The assistant: LlamaIndex index + retriever, LangChain generation
Features: multi-document search, per-document filter (metadata filters), conversational follow-ups, source citations, per-document summaries, save / load index.

In [ ]:
class DocAssistant:
    def __init__(self, chunk_size: int = 512, chunk_overlap: int = 64):
        setup_llamaindex(chunk_size, chunk_overlap)
        self.index = VectorStoreIndex(nodes=[])
        self.chunks = {}                      # file_name -> number of chunks
        self._chains = None

    @property
    def chains(self):
        if self._chains is None:
            self._chains = build_chains()
        return self._chains

    # ---- indexing (LlamaIndex) ----
    def add_documents(self, docs: List[Document]) -> int:
        nodes = Settings.node_parser.get_nodes_from_documents(docs)
        self.index.insert_nodes(nodes)
        for n in nodes:
            name = n.metadata.get("file_name", "unknown")
            self.chunks[name] = self.chunks.get(name, 0) + 1
        return len(nodes)

    def documents(self) -> dict:
        return dict(self.chunks)

    def save(self, path: str):
        self.index.storage_context.persist(persist_dir=path)

    def load(self, path: str):
        self.index = load_index_from_storage(StorageContext.from_defaults(persist_dir=path))
        self.chunks = {}
        for n in self.index.docstore.docs.values():
            name = n.metadata.get("file_name", "unknown")
            self.chunks[name] = self.chunks.get(name, 0) + 1

    # ---- retrieval (LlamaIndex retriever) ----
    def retrieve(self, question: str, doc_names: Optional[List[str]] = None, k: int = 4):
        filters = None
        if doc_names:
            filters = MetadataFilters(filters=[MetadataFilter(key="file_name", value=list(doc_names),
                                                               operator=FilterOperator.IN)])
        return self.index.as_retriever(similarity_top_k=k, filters=filters).retrieve(question)

    # ---- generation (LangChain) ----
    def ask(self, question: str, doc_names: Optional[List[str]] = None, k: int = 4, history=None) -> dict:
        if not self.chunks:
            return {"answer": "No documents are indexed yet. Upload a file or add a URL first.",
                    "found": False, "sources": [], "all_sources": [], "standalone_question": question}
        history = (history or [])[-6:]
        hist_txt = "\n".join(f"{role}: {text}" for role, text in history) or "(none)"
        standalone = question
        if history:
            standalone = run_with_retry(lambda: self.chains["condense"].invoke(
                {"history": hist_txt, "question": question})).strip() or question
        nodes = self.retrieve(standalone, doc_names, k)
        context = "\n\n".join(
            f"[{i}] (source: {n.metadata.get('file_name')}, page {n.metadata.get('page_label', '-')}) {n.get_content()}"
            for i, n in enumerate(nodes, 1))
        res = run_with_retry(lambda: self.chains["qa"].invoke(
            {"history": hist_txt, "context": context, "question": standalone}))
        all_sources = [{"id": i, "file": n.metadata.get("file_name"), "page": n.metadata.get("page_label", "-"),
                        "score": round(float(n.score), 3) if n.score is not None else None,
                        "snippet": n.get_content()[:300]} for i, n in enumerate(nodes, 1)]
        cited = [s for s in all_sources if s["id"] in res.cited_sources] or all_sources
        return {"answer": res.answer, "found": res.found_in_documents, "sources": cited,
                "all_sources": all_sources, "standalone_question": standalone}

    def summarize(self, doc_name: str, max_chunks: int = 8) -> str:
        nodes = [n for n in self.index.docstore.docs.values() if n.metadata.get("file_name") == doc_name]
        if not nodes:
            return f"Document '{doc_name}' not found."
        if len(nodes) > max_chunks:               # sample evenly to stay within API limits
            step = len(nodes) / max_chunks
            nodes = [nodes[int(i * step)] for i in range(max_chunks)]
        parts = run_with_retry(lambda: self.chains["map"].batch(
            [{"text": n.get_content()} for n in nodes], config={"max_concurrency": 2}))
        joined = "\n".join(f"- {p.strip()}" for p in parts)
        return run_with_retry(lambda: self.chains["reduce"].invoke({"name": doc_name, "parts": joined}))

## 5. Build the index
Three sample documents are created so the notebook runs anywhere (section 10 shows how to use your own PDFs). A small `chunk_size` is used here only because the sample files are short; the default for real documents is 512.

In [7]:
paths = write_sample_docs("sample_docs")
assistant = DocAssistant(chunk_size=128, chunk_overlap=20)
docs = load_files(paths)
print("Chunks indexed:", assistant.add_documents(docs))
print(assistant.documents())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunks indexed: 6
{'rolling_forecast_guide.txt': 2, 'variance_analysis_policy.txt': 2, 'ecommerce_kpi_handbook.txt': 2}


## 6. Helper to display answers

In [ ]:
def show(res):
    print("Q (used for retrieval):", res["standalone_question"])
    print("\nANSWER:", res["answer"])
    print("Found in documents:", res["found"])
    print("\nSOURCES:")
    for s in res["sources"]:
        print(f"  [{s['id']}] {s['file']} (page {s['page']}, score {s['score']}): {s['snippet'][:110]}...")

## 7. Question answering: single and multi-document

In [9]:
show(assistant.ask("How often is the rolling forecast refreshed?"))

Q (used for retrieval): How often is the rolling forecast refreshed?

ANSWER: The forecast is refreshed on the fifth working day of each month.
Found in documents: True

SOURCES:
  [1] rolling_forecast_guide.txt (page -, score 0.584): Acme Retail Rolling Forecast Guide.
Acme maintains a 12-month rolling forecast. After every month-end close th...


In [10]:
# Multi-document question: needs facts from two different files
show(assistant.ask("Which team supplies marketing spend for the forecast, and what CAC target should marketing stay below?", k=5))

Q (used for retrieval): Which team supplies marketing spend for the forecast, and what CAC target should marketing stay below?

ANSWER: The Marketing team supplies the marketing spend, and the CAC target is below INR 450.
Found in documents: True

SOURCES:
  [1] rolling_forecast_guide.txt (page -, score 0.396): Acme Retail Rolling Forecast Guide.
Acme maintains a 12-month rolling forecast. After every month-end close th...
  [3] ecommerce_kpi_handbook.txt (page -, score 0.337): Acme Retail E-commerce KPI Handbook.
Average order value (AOV) equals net revenue divided by the number of ord...


### 7.1 Search inside one document only (metadata filter)

In [11]:
show(assistant.ask("What are the targets?", doc_names=["ecommerce_kpi_handbook.txt"]))

Q (used for retrieval): What are the targets?

ANSWER: The handbook sets a target for customer acquisition cost (CAC) of below INR 450 and a target for contribution margin of 8 %. It also notes an internal benchmark conversion rate of 2.5 %.
Found in documents: True

SOURCES:
  [1] ecommerce_kpi_handbook.txt (page -, score 0.071): Acme Retail E-commerce KPI Handbook.
Average order value (AOV) equals net revenue divided by the number of ord...


### 7.2 Follow-up questions (conversation memory + question condensing)

In [12]:
history = []
q1 = "What is the investigation threshold for variances?"
r1 = assistant.ask(q1, history=history); show(r1)
history += [("user", q1), ("assistant", r1["answer"])]
print("\n" + "=" * 70 + "\n")
r2 = assistant.ask("And how soon must the commentary be written?", history=history); show(r2)

Q (used for retrieval): What is the investigation threshold for variances?

ANSWER: The investigation threshold is any line item that deviates by more than 5% from budget and by more than INR 10 lakh.
Found in documents: True

SOURCES:
  [1] variance_analysis_policy.txt (page -, score 0.577): Acme Retail Variance Analysis Policy.
Variance is calculated as actual minus budget. A variance is favourable ...


Q (used for retrieval): After a variance investigation threshold is met, how soon must the commentary be written?

ANSWER: The commentary must be written within three working days of the month‑end close.
Found in documents: True

SOURCES:
  [2] variance_analysis_policy.txt (page -, score 0.456): Acme Retail Variance Analysis Policy.
Variance is calculated as actual minus budget. A variance is favourable ...


### 7.3 A question that is NOT in the documents (hallucination guard)

In [13]:
show(assistant.ask("Who is the CEO of Acme Retail?"))

Q (used for retrieval): Who is the CEO of Acme Retail?

ANSWER: I could not find the CEO of Acme Retail in the provided documents.
Found in documents: False

SOURCES:
  [1] rolling_forecast_guide.txt (page -, score 0.453): Acme Retail Rolling Forecast Guide.
Acme maintains a 12-month rolling forecast. After every month-end close th...
  [2] ecommerce_kpi_handbook.txt (page -, score 0.403): Acme Retail E-commerce KPI Handbook.
Average order value (AOV) equals net revenue divided by the number of ord...
  [3] variance_analysis_policy.txt (page -, score 0.342): Acme Retail Variance Analysis Policy.
Variance is calculated as actual minus budget. A variance is favourable ...
  [4] ecommerce_kpi_handbook.txt (page -, score 0.212): The KPI review meeting is held every Monday. The Power BI dashboards are refreshed daily at 6 AM....


## 8. Summarise a document (LangChain map-reduce)

In [14]:
print(assistant.summarize("variance_analysis_policy.txt"))

**Overview:**  
Acme Retail’s variance analysis policy defines how variances are measured, investigated, and reported to ensure timely corrective action and accurate forecasting.

**Key facts and numbers**

- Variance = Actual – Budget; favourable variances increase profit, unfavourable variances reduce profit.  
- Any line item deviating **>5 %** from budget **and** exceeding **INR 10 lakh** triggers an investigation.  
- Variances are broken down into **volume, price, and mix** effects to identify root causes.  
- Each investigated variance must include a written commentary (cause, financial impact, action owner) and be submitted **within three working days** after month‑end close.  
- An unfavourable variance recorded for **three consecutive months** triggers a re‑forecast for that line item.


## 9. Mini evaluation: retrieval hit rate and answer correctness

In [ ]:
import re
import pandas as pd

norm = lambda t: re.sub(r"[^a-z0-9%]", "", t.lower())   # ignore spaces / dots / case

tests = [
    ("How often is the rolling forecast refreshed?", "rolling_forecast_guide.txt", "fifth working day"),
    ("Who must approve a forecast change above 2%?", "rolling_forecast_guide.txt", "Finance Director"),
    ("What is the variance investigation threshold?", "variance_analysis_policy.txt", "5%"),
    ("What is the CAC target?", "ecommerce_kpi_handbook.txt", "450"),
    ("When are the Power BI dashboards refreshed?", "ecommerce_kpi_handbook.txt", "6 AM"),
]
rows = []
for q, doc, keyword in tests:
    r = assistant.ask(q)
    rows.append({"question": q, "expected_doc": doc,
                 "retrieved_doc_ok": doc in {s["file"] for s in r["all_sources"]},
                 "answer_ok": norm(keyword) in norm(r["answer"])})
df = pd.DataFrame(rows)
display(df)
print("Retrieval hit rate:", df.retrieved_doc_ok.mean(), "| Answer accuracy:", df.answer_ok.mean())

## 10. Use your own documents (PDF / TXT / web page)
In Colab, run the upload cell and pick PDFs. In Jupyter / VS Code, put file paths in `my_paths`. Web pages: use `load_web(url)`.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()                       # choose one or more PDFs
    my_paths = list(uploaded.keys())
except Exception:
    my_paths = []                                   # e.g. ["my_report.pdf"]

if my_paths:
    print("Chunks added:", assistant.add_documents(load_files(my_paths)))
    print(assistant.documents())

In [ ]:
AR = "Acme_Retail_Annual_Report_FY2025-26.pdf"
QR = "Acme_Retail_Q2_Business_Review.pdf"

show(assistant.ask("What was the net revenue and contribution margin in FY2025-26?", doc_names=[AR]))
show(assistant.ask("How did the Q2 CAC compare with the full-year CAC and the target?", doc_names=[AR, QR], k=5))
show(assistant.ask("Why was marketing spend above budget in Q2, and was it investigated?", doc_names=[QR]))
show(assistant.ask("What is the dividend per share for FY2024-25?", doc_names=[AR]))                                         # should say not found
print(assistant.summarize(AR))

In [18]:
# Optional: add a web page (needs internet)
try:
    n = assistant.add_documents(load_web("https://en.wikipedia.org/wiki/Retrieval-augmented_generation"))
    print("Web page chunks:", n)
    show(assistant.ask("What problem does retrieval-augmented generation solve?"))
except Exception as e:
    print("Skipped web page:", e)

Web page chunks: 81
Q (used for retrieval): What problem does retrieval-augmented generation solve?

ANSWER: Retrieval‑augmented generation addresses the problem that large language models can become out‑of‑date or hallucinate facts by allowing them to pull in fresh, external information at inference time, so that generated responses are grounded in up‑to‑date data rather than only the model’s static training set.
Found in documents: True

SOURCES:
  [1] Retrieval-augmented generation - Wikipedia (page -, score 0.625): Ars Technica
 notes that "when new information becomes available, rather than having to retrain the model, all...
  [4] Retrieval-augmented generation - Wikipedia (page -, score 0.597): [
1
]

The model feeds this relevant retrieved information into the LLM via 
prompt engineering
 of the user's...


## 11. Save and reload the index (no re-embedding needed)

In [19]:
assistant.save("storage")
restored = DocAssistant()
restored.load("storage")
print("Reloaded:", restored.documents())

Reloaded: {'rolling_forecast_guide.txt': 2, 'variance_analysis_policy.txt': 2, 'ecommerce_kpi_handbook.txt': 2, 'Acme_Retail_Q2_Business_Review.pdf': 8, 'Acme_Retail_Annual_Report_FY2025-26.pdf': 16, 'Retrieval-augmented generation - Wikipedia': 81}


## 12. Limitations and future scope
- Retrieval uses vector similarity only; hybrid (BM25 + vector) search and a re-ranker would improve precision.
- Scanned PDFs need OCR before indexing.
- The LLM follows the 'answer only from context' instruction but can still err; the sources are shown so answers can be verified.
- In-memory index: use Chroma / FAISS for large collections.
- Deployment: see `streamlit_app.py` (Streamlit Community Cloud).